# Context Store Setup
Validate configuration and credentials, create approval folders, and optionally create the BigQuery context tables and views.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys

ROOT = Path.cwd()
if not (ROOT / 'config').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import config_summary, load_app_config
from dq_agent.context_store import ensure_context_objects
from dq_agent.context_utils import configure_workflow_logging, context_workflow_paths, logged_step

## Editable values

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
APPLY_BIGQUERY_DDL = False  # Review config/project.yaml before changing this to True

In [ ]:
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)
with logged_step(logger, paths['checkpoint'], 'LOAD_CONFIGURATION', project_file='config/project.yaml'):
    print(config_summary(config))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'VALIDATE_CONTEXT_FILES'):
    required = [
        config.path(config.project.table_mappings),
        config.path(config.project.business_context),
        config.path(config.project.human_tests),
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Missing inputs: {missing}')
    print('Approval folders:', {name: str(path) for name, path in paths.items() if name in {'pending','processed','approved','rejected','errors'}})

In [ ]:
if APPLY_BIGQUERY_DDL:
    with logged_step(logger, paths['checkpoint'], 'WRITE_CONTEXT_STAGING'):
        objects = ensure_context_objects(config, logger)
        display(objects)
else:
    print('DDL skipped. Set context_store.enabled=true and APPLY_BIGQUERY_DDL=True when ready.')

## Manual verification
1. Confirm `checkpoint.json` reports `VALIDATE_CONTEXT_FILES`.
2. After enabling DDL, confirm all three tables and two views are listed.
3. Open `workflow.log` beside this run's checkpoint to inspect step-level logging.